# 05 · Nested Models, Unions, Discriminated Unions, Generics & Recursive Models

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 50 min

### 🎯 What you will learn
- Nesting models inside models (and lists/dicts of models)
- Union types: **smart mode** vs left-to-right
- ✅ **Discriminated (tagged) unions** with `Literal` + `Field(discriminator=...)`, and callable `Discriminator`
- **Generic** models (`Response[T]`, `Page[T]`)
- **Recursive** / self-referencing models (trees, comments)
- `RootModel` for top-level lists/values
- Model inheritance and composition

In [1]:
from datetime import datetime
from typing import Annotated, Literal, Any
from pydantic import BaseModel, Field, ValidationError, RootModel, Discriminator, Tag

## 1. Nested models

In [2]:
class Address(BaseModel):
    street: str
    city: str
    pincode: str = Field(pattern=r"^\d{6}$")

class OrderLine(BaseModel):
    sku: str
    qty: int = Field(gt=0)
    unit_price: float = Field(gt=0)

class Order(BaseModel):
    id: int
    shipping: Address
    billing: Address | None = None
    lines: list[OrderLine] = Field(min_length=1)
    metadata: dict[str, str] = {}

    @property
    def total(self) -> float:
        return sum(l.qty * l.unit_price for l in self.lines)

data = {
    "id": 1042,
    "shipping": {"street": "MG Road 1", "city": "Pune", "pincode": "411001"},
    "lines": [{"sku": "PEN", "qty": 2, "unit_price": 10}, {"sku": "BOOK", "qty": "1", "unit_price": "250"}],
}
order = Order.model_validate(data)
print(order.shipping.city, order.lines[1].qty, order.total)

try:
    Order.model_validate({**data, "shipping": {"street": "x", "city": "y", "pincode": "12"},
                          "lines": [{"sku": "A", "qty": 0, "unit_price": 1}]})
except ValidationError as e:
    for err in e.errors():
        print(err["loc"], "→", err["msg"])           # locations show the full path

Pune 1 270.0
('shipping', 'pincode') → String should match pattern '^\d{6}$'
('lines', 0, 'qty') → Input should be greater than 0


## 2. Unions

By default Pydantic uses **smart mode**: it picks the union member that matches the input **best** (exact type first), not simply the first that validates:

In [3]:
class Flexible(BaseModel):
    value: int | float | str

for v in [1, 1.5, "1", "hello"]:
    m = Flexible(value=v)
    print(f"{v!r:<8} → {m.value!r} ({type(m.value).__name__})")

1        → 1 (int)
1.5      → 1.5 (float)
'1'      → '1' (str)
'hello'  → 'hello' (str)


`Field(union_mode="left_to_right")` tries members in order instead.

## 3. ✅ Discriminated unions

When a union contains **models**, tell Pydantic which field identifies the type. It then validates against exactly one member: faster, and with much clearer errors.

In [4]:
class CardPayment(BaseModel):
    method: Literal["card"]
    card_last4: str = Field(pattern=r"^\d{4}$")

class UPIPayment(BaseModel):
    method: Literal["upi"]
    upi_id: str = Field(pattern=r"^[\w.\-]+@[\w]+$")

class CashPayment(BaseModel):
    method: Literal["cash"]
    change_due: float = 0

Payment = Annotated[CardPayment | UPIPayment | CashPayment, Field(discriminator="method")]

class Checkout(BaseModel):
    order_id: int
    payments: list[Payment]

c = Checkout.model_validate({"order_id": 1, "payments": [
    {"method": "upi", "upi_id": "asha@okbank"},
    {"method": "card", "card_last4": "4242"},
    {"method": "cash"},
]})
print([type(p).__name__ for p in c.payments])

try:
    Checkout.model_validate({"order_id": 1, "payments": [{"method": "upi", "upi_id": "bad id"}, {"method": "bitcoin"}]})
except ValidationError as e:
    for err in e.errors():
        print(err["loc"], "→", err["msg"])

['UPIPayment', 'CardPayment', 'CashPayment']
('payments', 0, 'upi', 'upi_id') → String should match pattern '^[\w.\-]+@[\w]+$'
('payments', 1) → Input tag 'bitcoin' found using 'method' does not match any of the expected tags: 'card', 'upi', 'cash'


Errors point to the **right** member (`payments.0.upi.upi_id`) instead of listing failures for every possible type.

### Event processing with discriminated unions

In [5]:
class UserCreated(BaseModel):
    type: Literal["user.created"]
    user_id: int
    email: str

class OrderPaid(BaseModel):
    type: Literal["order.paid"]
    order_id: int
    amount: float

Event = Annotated[UserCreated | OrderPaid, Field(discriminator="type")]

class Webhook(BaseModel):
    received_at: datetime
    event: Event

def handle(raw: str) -> str:
    hook = Webhook.model_validate_json(raw)
    match hook.event:
        case UserCreated(email=email):
            return f"send welcome mail to {email}"
        case OrderPaid(order_id=oid, amount=amt):
            return f"ship order {oid} (₹{amt})"

print(handle('{"received_at": "2026-10-20T10:00:00Z", "event": {"type": "order.paid", "order_id": 9, "amount": 499}}'))

ship order 9 (₹499.0)


### Callable discriminators (no shared field)

In [6]:
def shape_kind(v: Any) -> str:
    keys = v.keys() if isinstance(v, dict) else vars(v).keys()
    return "circle" if "radius" in keys else "rect"

class Circle(BaseModel):
    radius: float
class Rect(BaseModel):
    w: float
    h: float

Shape = Annotated[Annotated[Circle, Tag("circle")] | Annotated[Rect, Tag("rect")], Discriminator(shape_kind)]

class Drawing(BaseModel):
    shapes: list[Shape]

print(Drawing(shapes=[{"radius": 1}, {"w": 2, "h": 3}]).shapes)

[Circle(radius=1.0), Rect(w=2.0, h=3.0)]


## 4. Generic models

In [7]:
class Page[T](BaseModel):
    items: list[T]
    page: int = 1
    total: int

    @property
    def pages(self) -> int:
        return -(-self.total // 10)

class Product(BaseModel):
    sku: str
    price: float

class ApiResponse[T](BaseModel):
    ok: bool = True
    data: T | None = None
    error: str | None = None

resp = ApiResponse[Page[Product]].model_validate(
    {"data": {"items": [{"sku": "A", "price": "9.5"}], "total": 25}})
print(resp.data.items[0], resp.data.pages)
print(ApiResponse[int](data="42").data)
try:
    ApiResponse[Page[Product]].model_validate({"data": {"items": [{"sku": "A"}], "total": 1}})
except ValidationError as e:
    print(e.errors()[0]["loc"], e.errors()[0]["msg"])

sku='A' price=9.5 3
42
('data', 'items', 0, 'price') Field required


> ⚠️ Pydantic v1 needed `GenericModel` from `pydantic.generics` + `TypeVar`; in v2 any `BaseModel` can be generic (here with the 3.12 `class Page[T]` syntax).

## 5. Recursive models

In [8]:
class Comment(BaseModel):
    author: str
    text: str
    replies: list["Comment"] = []              # refers to itself

    def count(self) -> int:
        return 1 + sum(r.count() for r in self.replies)

thread = Comment.model_validate({
    "author": "asha", "text": "Great lesson!",
    "replies": [{"author": "ravi", "text": "Agreed", "replies": [{"author": "meena", "text": "+1"}]},
                {"author": "john", "text": "Thanks"}],
})
print(thread.count(), thread.replies[0].replies[0].author)

class Category(BaseModel):
    name: str
    children: list[Category] = []             # Python 3.14+: no quotes needed (deferred annotations)

print(Category.model_validate({"name": "root", "children": [{"name": "books"}]}))

4 meena
name='root' children=[Category(name='books', children=[])]


If a model references a class defined **later**, call `Model.model_rebuild()` after defining it.

## 6. `RootModel`: when the data isn't an object

In [9]:
class Tags(RootModel[list[str]]):
    def __iter__(self):
        return iter(self.root)

class Prices(RootModel[dict[str, float]]):
    pass

tags = Tags.model_validate_json('["python", "pydantic"]')
print(tags.root, list(tags), tags.model_dump_json())
print(Prices({"pen": "10.5"}).root)

['python', 'pydantic'] ['python', 'pydantic'] ["python","pydantic"]
{'pen': 10.5}


(For one-off validation of plain types, `TypeAdapter` (lesson 07) is often simpler.)

## 7. Inheritance & composition

In [10]:
class TimestampMixin(BaseModel):
    created_at: datetime = Field(default_factory=datetime.now)
    updated_at: datetime | None = None

class UserBase(BaseModel):
    email: str
    name: str

class UserCreate(UserBase):            # input from clients
    password: str = Field(min_length=8)

class UserPublic(UserBase, TimestampMixin):   # output to clients (no password!)
    id: int

print(list(UserCreate.model_fields), list(UserPublic.model_fields))

['email', 'name', 'password'] ['created_at', 'updated_at', 'email', 'name', 'id']


The pattern **Base / Create / Update / Public** models is standard in FastAPI apps (lesson 10).

## 📝 Summary
- Models nest freely; error locations show the full path.
- Unions use smart mode; ✅ use **discriminated unions** (`Literal` tag + `Field(discriminator=...)`) for model unions.
- Generic models (`class Page[T](BaseModel)`) and recursive models just work; `RootModel` validates non-object data.
- Share fields via inheritance; separate input and output models.

## 🏋️ Practice
1. Model a notification system with a discriminated union of `EmailNotification`, `SMSNotification`, `PushNotification` (field `channel`).
2. Create a generic `Result[T]` with `value: T | None` and `errors: list[str]`, and use it with `Result[list[int]]`.
3. Model a file-system tree (`Folder` with `files: list[File]` and `folders: list[Folder]`) and compute the total size recursively.